# 🧠 AI Logic Engine: Knowledge Ingestion (Final Speed & Debug Pass)

### 🚀 Update Log (v4.2.1):
- **Unicode Support:** မြန်မာစာ ID များကိုပါ လက်ခံနိုင်အောင် ပြင်ထားပါသည်။
- **Heartbeat Logs:** AI အလုပ်လုပ်နေလား၊ စာထုတ်နေလားဆိုတာကို အဆင့်တိုင်းမှာ ပြပေးပါမည်။
- **Scope:** Small Talk, Manners, General Knowledge ဦးစားပေး။

In [ ]:
# @title 🛠️ Step 1: Install & Imports
!pip install -q firebase-admin transformers accelerate bitsandbytes torch tqdm

import os, json, re, torch, time
from tqdm.auto import tqdm
import firebase_admin
from firebase_admin import credentials, firestore
from google.colab import files
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

print("✅ Core Libraries Loaded.")

In [ ]:
# @title 🔑 Step 2: Firestore Connection
DATABASE_ID = 'ai-studio-09d97652-b1c3-4b1a-b63e-31d8a38be4c2'

if not firebase_admin._apps:
    print("Service Account Key (JSON) ကို Upload တင်ပေးပါ...")
    uploaded = files.upload()
    if not uploaded:
        raise ValueError("❌ JSON key လိုအပ်ပါသည်။")
    filename = list(uploaded.keys())[0]
    cred = credentials.Certificate(filename)
    firebase_admin.initialize_app(cred)

db = firestore.client(database_id=DATABASE_ID)
print(f"✅ Firestore Synced: {DATABASE_ID}")

In [ ]:
# @title 🧬 Step 3: Logic Engine Config
STATE_FILE = "ingestion_state_v4_2.json"

def clean_id(text):
    if not text: return ""
    text = str(text).lower().strip()
    # Support Unicode characters but replace spaces/slashes with underscores
    return re.sub(r'[\s/]+', '_', text).strip('_')

def normalize_verb(v):
    v = str(v).strip().lower()
    is_a_set = ['is_a', 'is a', 'is type of', 'belongs to', 'category of', 'member of']
    if any(p == v for p in is_a_set): return 'is_a'
    return v.replace(' ', '_')

def save_state(count):
    with open(STATE_FILE, "w") as f: json.dump({"count": count}, f)

def load_state():
    if os.path.exists(STATE_FILE):
        try:
            with open(STATE_FILE, "r") as f: return json.load(f).get("count", 0)
        except: pass
    return 0

print("✅ Ingestion Logic Armored.")

In [ ]:
# @title 🤖 Step 4: Load Faster Model (Qwen 1.5B)
model_id = "Qwen/Qwen2.5-1.5B-Instruct"
print(f"⏳ Loading {model_id} into T4 GPU...")

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True, 
    bnb_4bit_compute_dtype=torch.float16, 
    bnb_4bit_quant_type="nf4"
)

tokenizer = AutoTokenizer.from_pretrained(model_id)
if tokenizer.pad_token is None: 
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_id, 
    quantization_config=bnb_config, 
    device_map="auto"
)
model.eval()
print("✅ AI Model Active.")

In [ ]:
# @title 🚀 Step 5: Start Continuous Ingestion
TARGET_FACT_COUNT = 1000000 # @param {type:"number"}
CATEGORIES = [
    "Small Talk Greetings", "Social Manners", "General Etiquette", 
    "Common Polite Phrases", "Everyday Humor", "Basic World Facts",
    "Meeting New People", "Office Conversations", "Travel Safety Tips"
]

current_count = load_state()
pbar = tqdm(initial=current_count, total=TARGET_FACT_COUNT, desc="Syncing Symbols")

print(f"\n🔥 INGESTION STARTING (Target: {TARGET_FACT_COUNT})")
print("--------------------------------------------------")

while current_count < TARGET_FACT_COUNT:
    try:
        cat = CATEGORIES[current_count % len(CATEGORIES)]
        
        # Faster & More Forced Prompt
        prompt = f"Extract 20 unique factual triplets about {cat}. Format: S|V|O|Sentence. Strictly no chat, only pipe-separated lines."
        messages = [
            {"role": "system", "content": "Output S|V|O|Sentence ONLY. No intros, no headers."},
            {"role": "user", "content": prompt}
        ]
        
        # HEARTBEAT 1
        it = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inp = tokenizer(it, return_tensors="pt", padding=True).to(model.device)
        
        print(f"⏳ AI is thinking about: {cat}...", end="\r")
        with torch.no_grad():
            out = model.generate(
                inp.input_ids, 
                attention_mask=inp.attention_mask, 
                max_new_tokens=600, 
                do_sample=True, 
                temperature=0.7,
                pad_token_id=tokenizer.pad_token_id
            )
        
        # HEARTBEAT 2
        response = tokenizer.decode(out[0][inp.input_ids.shape[1]:], skip_special_tokens=True).strip()
        print(f"✅ AI responded. Parsing data...       ", end="\r")
        
        if not response: 
            print("⚠️ Empty response from AI.             ")
            continue
        
        batch = db.batch()
        added_this_loop = 0
        
        for line in response.split('\n'):
            if '|' not in line: continue
            parts = [p.strip() for p in line.split('|')]
            
            if len(parts) >= 3:
                s, v, o = parts[0], normalize_verb(parts[1]), parts[2]
                sid, oid = clean_id(s), clean_id(o)
                
                if sid and oid and sid != oid:
                    node_ref = db.collection('nodes').document(sid)
                    data = {'name': s, 'updatedAt': firestore.SERVER_TIMESTAMP}
                    
                    if v == 'is_a':
                        data['groups'] = firestore.ArrayUnion([oid])
                    else:
                        sent = parts[3] if len(parts) > 3 else f"{s} {v.replace('_',' ')} {o}."
                        data['relations'] = firestore.ArrayUnion([{
                            'verb': v, 'targetId': oid, 'sentence': sent, 'weight': 100
                        }])
                    
                    batch.set(node_ref, data, merge=True)
                    added_this_loop += 1
        
        if added_this_loop > 0:
            batch.commit()
            current_count += added_this_loop
            pbar.update(added_this_loop)
            save_state(current_count)
            # print(f"✅ Synced {added_this_loop} facts.")
        else:
            print(f"⚠️ No valid symbols found for {cat}.   ")

        if current_count % 30 == 0: torch.cuda.empty_cache()
        
    except Exception as e:
        print(f"\n❌ Loop Error: {e}. Cooling down 5s...")
        time.sleep(5)
    except KeyboardInterrupt:
        print("\n🛑 Manually Stopped.")
        break

pbar.close()
print(f"✅ Finished. Current Knowledge Base Total: {current_count}")